# 8장 — 기출 유출: 오염과 벤치마크의 수명 🕵️

"유명 벤치마크 +17점 도약!" 문항 수도 p값도 완벽한데 **가짜**일 수 있습니다.
평가 문항이 훈련 데이터에 새어 들어가는 **오염(contamination)**은 점수를 위로만 미는
**체계적 편향**이라, 6·7장의 통계 도구(오차범위·p값)로는 잡히지 않아요.

신의 시점 실험: 진짜 실력 60%, 유출된 문항은 암기로 95% 정답.

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 여기서 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch8')

## 1. 오염 다이얼: 점수가 부푸는 구조

관측 점수 = f·95 + (1−f)·60 (f = 오염률). 변형판·신규 홀드아웃은 암기가 안 통해 60 그대로.

In [ ]:
TRUE_P, MEM_P = 60.0, 95.0

def observed(f):
    """오염률 f 에서의 기존 벤치마크 관측 점수."""
    return f * MEM_P + (1 - f) * TRUE_P

print(' 오염률 | 기존 벤치마크 | 변형판/홀드아웃 | 부풀림')
print('-' * 52)
for pct in [0, 20, 40, 60, 80, 100]:
    f = pct / 100
    ob = observed(f)
    print(f'{pct:>5}% | {ob:11.1f} | {TRUE_P:13.1f} | +{ob - TRUE_P:.1f}%p')

assert observed(0.4) == 74.0, '오염률 40% → 74점 (StepQuiz: 38+36)'
assert observed(0.5) == 77.5
assert observed(1.0) == 95.0
print()
print('✔ 진짜 실력은 내내 60. 헤드라인의 도약은 유출 문항의 암기였다')

## 2. 탐정의 산수: 낙차로 오염률 역산

부풀림은 편향이라 문항 수로 못 잡지만, **지문**이 남습니다:
낙차 Δ = 기존 − 변형 = f × 35%p → **f̂ = Δ / 35**.

In [ ]:
def f_hat(delta):
    return delta / (MEM_P - TRUE_P)

for delta in [7, 14, 21, 28]:
    print(f'낙차 Δ = {delta:>2}%p → 추정 오염률 f̂ = {f_hat(delta)*100:.0f}%')

assert round(f_hat(14) * 100) == 40
assert round(f_hat(21) * 100) == 60
# 역산의 자기 일관성 — 다이얼 f 로 만든 낙차를 되돌리면 f 가 나온다
for f in [0.15, 0.4, 0.85]:
    assert abs(f_hat(observed(f) - TRUE_P) - f) < 1e-12
print()
print('✔ 낙차가 오염률을 자백한다. 변형판 시험이 탐지 도구인 이유')

## 3. 그림: 편향은 문항 수로 줄지 않는다

왼쪽: 오염률에 따른 세 시험의 점수. 오른쪽: 문항 수를 늘리면 **소음(±)은 줄지만
부풀림(편향)은 그대로**라는 결정적 대비 (6장의 오차범위 공식 재사용).

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False
except Exception:
    pass

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

fs = np.linspace(0, 1, 101)
ax1.plot(fs * 100, [observed(f) for f in fs], color='#fb7185', lw=2, label='기존 벤치마크 (오염)')
ax1.axhline(TRUE_P, color='#10b981', lw=2, label='변형판·홀드아웃 (진짜 실력)')
ax1.set_xlabel('오염률 f (%)'); ax1.set_ylabel('점수')
ax1.set_title('오염 다이얼: 부풀림 = f × 35%p')
ax1.legend(fontsize=8)

ns = np.array([50, 100, 250, 1000, 5000, 10000])
bias = observed(0.4) - TRUE_P                       # 오염 40%의 부풀림(고정)
noise = 1.96 * np.sqrt(0.74 * 0.26 / ns) * 100      # 6장의 오차범위(줄어듦)
ax2.semilogx(ns, [bias] * len(ns), color='#fb7185', lw=2, label='편향(오염 40%): 그대로')
ax2.semilogx(ns, noise, color='#6366f1', lw=2, label='소음(±오차범위): √n 으로 감소')
ax2.set_xlabel('문항 수 n'); ax2.set_ylabel('%p')
ax2.set_title('오차범위의 사각지대: 편향은 안 줄어든다')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

assert abs(noise[-1] - 0.86) < 0.01 and bias == 14.0
print('✔ n=10,000이면 소음은 ±0.9%p 로 줄지만 부풀림 14%p 는 요지부동. 통계가 못 잡는 적')

## 정리: 오늘 가져갈 세 문장

1. **점수 = 실력 + 소음 + 편향(오염)**: 오염은 √n 으로 줄지 않는 체계적 편향.
2. 탐정의 산수: **변형판과의 낙차 Δ = f × 35%p**. 낙차가 오염률을 자백한다.
3. 점수 급등 의심 목록 4종: 불균형 급등 / 변형 취약 / 공개일 vs 컷오프 / 문항 암송.
   감정 절차는 6장(오차범위) → 7장(짝지은 비교·p값) → 8장(오염 검사) 직렬.

다음 **9장(AI 채점관의 성적표)**: 채점자마저 AI일 때의 편향(위치·길이·자기 선호)과
심판 감정법(3부 10장 상관 복습).